# 📚 Regresión Múltiple y Logarítmica con Datos de Venezuela
### Notebook para principiantes — Ciencias Sociales y Economía

---

## ¿Qué vamos a hacer?

En el notebook anterior aprendimos la **regresión lineal simple**: una variable X predice una variable Y.  
Ahora vamos a aprender dos modelos más potentes:

| Modelo | ¿Qué hace? |
|---|---|
| **Regresión Múltiple** | Usa **varias variables X** para predecir Y con más precisión |
| **Regresión Logarítmica** | Modela relaciones que **no son una línea recta**, sino una curva |

---

## Preguntas que vamos a responder

> **Regresión Múltiple:**  
> ¿Podemos predecir mejor la matrícula universitaria si usamos el gasto en educación, el PIB per cápita **y** las becas otorgadas juntos?

> **Regresión Logarítmica:**  
> ¿La relación entre el PIB per cápita y la matrícula universitaria sigue una **curva** en lugar de una línea recta?

---

### Variables del dataset

| Variable | Descripción |
|---|---|
| `Gasto_Educacion_%PIB` | % del PIB que el gobierno destina a educación |
| `PIB_per_Capita_USD` | Producción económica por habitante en dólares |
| `Becas_Otorgadas_Miles` | Número de becas entregadas (en miles) |
| `Tasa_Desempleo_%` | Porcentaje de la población sin empleo |
| `Matricula_Universitaria_%` | % de jóvenes en edad universitaria matriculados (**Y**) |

## Paso 1 — Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

print('✅ Librerías cargadas correctamente')

## Paso 2 — Cargar los datos

Usamos el mismo dataset de Venezuela (2000–2023).  
**Fuentes:** UNESCO, CEPAL, Banco Mundial, INE Venezuela

In [ ]:
datos = {
    'Año':                       [2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,
                                   2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,
                                   2020,2021,2022,2023],

    'Gasto_Educacion_%PIB':      [4.1,4.3,4.0,3.8,4.5,5.1,5.6,6.2,6.0,5.8,
                                   5.5,5.7,6.1,5.9,5.4,4.9,4.1,3.5,2.8,2.1,
                                   1.9,2.0,2.2,2.3],

    'Matricula_Universitaria_%': [26.4,27.1,27.8,28.5,30.2,33.4,36.8,39.1,40.3,41.2,
                                   42.0,43.1,44.5,44.8,44.2,43.1,40.8,37.5,33.2,29.4,
                                   26.8,25.1,24.3,23.8],

    'PIB_per_Capita_USD':        [4731,5044,4237,3587,4799,6063,7342,8335,9459,8613,
                                   9067,10526,11490,10170,9215,7737,5941,4038,2548,1651,
                                   1424,1589,1867,2067],

    'Tasa_Desempleo_%':          [13.9,13.3,15.8,18.0,15.3,12.4,10.0,8.5,7.3,7.9,
                                    8.5, 8.2, 7.8, 7.5, 7.2, 7.4, 7.3,7.0,6.8,7.0,
                                    7.2, 7.5, 5.5, 5.2],

    'Becas_Otorgadas_Miles':     [18.2,19.5,17.8,16.1,22.4,28.7,34.2,39.8,42.1,40.3,
                                   38.9,41.5,44.8,43.2,41.7,38.4,31.2,25.6,18.9,13.4,
                                   11.2,10.8,11.5,12.1]
}

df = pd.DataFrame(datos)
print(f'Dataset cargado: {df.shape[0]} filas y {df.shape[1]} columnas')
df

## Paso 3 — Explorar y limpiar los datos

Revisamos el estado del dataset y aplicamos las mismas herramientas de limpieza del notebook anterior.

In [ ]:
# Revisión general
print('Tipos de datos y valores nulos:')
df.info()
print()
print(f'Valores nulos  : {df.isnull().sum().sum()}')
print(f'Filas duplicadas: {df.duplicated().sum()}')

In [ ]:
# Introducimos errores intencionales para practicar la limpieza
df_sucio = df.copy()
df_sucio.loc[3,  'Gasto_Educacion_%PIB']      = None
df_sucio.loc[14, 'Becas_Otorgadas_Miles']      = -999
df_sucio.loc[19, 'PIB_per_Capita_USD']         = -999
df_sucio = pd.concat([df_sucio, df_sucio.iloc[[0]]], ignore_index=True)
df_sucio = df_sucio.sample(frac=1, random_state=7).reset_index(drop=True)

print(f'Dataset sucio — filas: {len(df_sucio)} | nulos: {df_sucio.isnull().sum().sum()} | duplicados: {df_sucio.duplicated().sum()}')

# Limpieza paso a paso
df_limpio = df_sucio.copy()
df_limpio = df_limpio.replace(-999, None)
df_limpio = df_limpio.dropna()
df_limpio = df_limpio.drop_duplicates()
df_limpio = df_limpio.sort_values('Año').reset_index(drop=True)

print(f'Dataset limpio — filas: {len(df_limpio)} | nulos: {df_limpio.isnull().sum().sum()} | duplicados: {df_limpio.duplicated().sum()}')
df_limpio

## Paso 4 — Análisis estadístico

Calculamos las medidas de tendencia central y dispersión para las variables principales.

In [ ]:
# Estadísticas de Matrícula Universitaria (Y)
matricula = df_limpio['Matricula_Universitaria_%']

print('📌 Matrícula Universitaria (%) — Variable Y')
print(f'  Media              : {matricula.mean():.2f}%')
print(f'  Mediana            : {matricula.median():.2f}%')
print(f'  Desviación estándar: {matricula.std():.2f}%')
print(f'  Mínimo             : {matricula.min():.2f}%')
print(f'  Máximo             : {matricula.max():.2f}%')
print(f'  Rango              : {matricula.max() - matricula.min():.2f}%')

In [ ]:
# Estadísticas del PIB per cápita
pib = df_limpio['PIB_per_Capita_USD']

print('📌 PIB per Cápita (USD)')
print(f'  Media              : {pib.mean():.2f} USD')
print(f'  Mediana            : {pib.median():.2f} USD')
print(f'  Desviación estándar: {pib.std():.2f} USD')
print(f'  Mínimo             : {pib.min():.2f} USD')
print(f'  Máximo             : {pib.max():.2f} USD')
print(f'  Rango              : {pib.max() - pib.min():.2f} USD')

In [ ]:
# Resumen de todas las variables
print('Resumen estadístico completo:')
df_limpio.describe().round(2)

## Paso 5 — Matriz de correlación

Revisamos cómo se relacionan todas las variables entre sí antes de construir los modelos.

> 💡 Un valor de **r cercano a +1** significa que las dos variables suben y bajan juntas.  
> Un valor **cercano a -1** significa que cuando una sube, la otra baja.

In [ ]:
correlacion = df_limpio.drop(columns='Año').corr(method='pearson').round(2)

plt.figure(figsize=(8, 6))
sns.heatmap(
    correlacion,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    annot_kws={'size': 11}
)
plt.title('Matriz de Correlación de Pearson (r)\nVenezuela 2000–2023', fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print('Variables con mayor correlación con Matrícula Universitaria:')
print(correlacion['Matricula_Universitaria_%'].drop('Matricula_Universitaria_%').sort_values(ascending=False))

---
# 🔵 PARTE A — Regresión Lineal Múltiple

## ¿Qué es la Regresión Múltiple?

En la regresión lineal **simple** usábamos una sola variable X:
$$\hat{Y} = \beta_0 + \beta_1 \cdot X$$

En la regresión **múltiple** usamos **varias variables X** al mismo tiempo:
$$\hat{Y} = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3$$

Cada $\beta$ indica cuánto aporta su variable a la predicción, **manteniendo las demás constantes**.

### Variables que usaremos
- $X_1$ = Gasto en Educación (% PIB)
- $X_2$ = PIB per Cápita (USD)
- $X_3$ = Becas Otorgadas (miles)
- $\hat{Y}$ = Matrícula Universitaria (%)

## Paso 6 — Construcción del modelo múltiple

### Matemáticamente

En la regresión múltiple no calculamos los coeficientes a mano como antes, porque la fórmula requiere **álgebra matricial**. La forma compacta es:

$$\boldsymbol{\beta} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{Y}$$

Donde $\mathbf{X}$ es la matriz de todas las variables predictoras y $\mathbf{Y}$ el vector de respuesta.  
Afortunadamente, **scikit-learn hace este cálculo automáticamente** por nosotros.

In [ ]:
# Definimos las variables predictoras (X) y la variable objetivo (Y)
X_mult = df_limpio[['Gasto_Educacion_%PIB', 'PIB_per_Capita_USD', 'Becas_Otorgadas_Miles']].values
Y      = df_limpio['Matricula_Universitaria_%'].values

# Creamos y entrenamos el modelo
modelo_mult = LinearRegression()
modelo_mult.fit(X_mult, Y)

# Obtenemos los coeficientes
b0_m = modelo_mult.intercept_
b1_m, b2_m, b3_m = modelo_mult.coef_

# Calculamos las predicciones y el R²
Y_pred_mult = modelo_mult.predict(X_mult)
r2_mult     = r2_score(Y, Y_pred_mult)

print('Coeficientes del modelo múltiple:')
print(f'  β₀ (intercepto)               = {b0_m:.4f}')
print(f'  β₁ (Gasto Educación % PIB)    = {b1_m:.4f}')
print(f'  β₂ (PIB per Cápita USD)       = {b2_m:.6f}')
print(f'  β₃ (Becas Otorgadas miles)    = {b3_m:.4f}')
print()
print(f'Ecuación:')
print(f'  Ŷ = {b0_m:.4f} + {b1_m:.4f}·X₁ + {b2_m:.6f}·X₂ + {b3_m:.4f}·X₃')
print()
print(f'R² del modelo múltiple = {r2_mult:.4f}')
print(f'El modelo explica el {r2_mult*100:.1f}% de la variabilidad de la matrícula.')

In [ ]:
# Interpretación de los coeficientes
print('Interpretación de los coeficientes:')
print()
print(f'  β₁ = {b1_m:.4f}')
print(f'  → Por cada 1% adicional del PIB invertido en educación,')
print(f'    la matrícula sube {b1_m:.2f} puntos porcentuales')
print(f'    (manteniendo PIB y Becas constantes).')
print()
print(f'  β₂ = {b2_m:.6f}')
print(f'  → Por cada 1 USD adicional de PIB per cápita,')
print(f'    la matrícula cambia {b2_m:.6f} puntos porcentuales')
print(f'    (efecto pequeño porque la escala del PIB es grande).')
print()
print(f'  β₃ = {b3_m:.4f}')
print(f'  → Por cada 1.000 becas adicionales otorgadas,')
print(f'    la matrícula sube {b3_m:.4f} puntos porcentuales')
print(f'    (manteniendo las otras variables constantes).')

## Paso 7 — Comparar el modelo múltiple con el simple

¿Realmente mejora usar tres variables en vez de una?  
Comparamos el R² de ambos modelos.

In [ ]:
# Modelo simple (solo Gasto en Educación) para comparar
X_simple = df_limpio[['Gasto_Educacion_%PIB']].values
modelo_simple = LinearRegression()
modelo_simple.fit(X_simple, Y)
r2_simple = r2_score(Y, modelo_simple.predict(X_simple))

print('Comparación de modelos:')
print(f'  Modelo simple   (1 variable)  R² = {r2_simple:.4f}  → explica el {r2_simple*100:.1f}%')
print(f'  Modelo múltiple (3 variables) R² = {r2_mult:.4f}  → explica el {r2_mult*100:.1f}%')
print()
mejora = (r2_mult - r2_simple) * 100
print(f'  Mejora al agregar más variables: +{mejora:.1f} puntos porcentuales de R²')

In [ ]:
# Gráfico: valores reales vs predichos por el modelo múltiple
plt.figure(figsize=(10, 6))

# Línea de referencia perfecta (si el modelo fuera perfecto, todos los puntos estarían aquí)
limite = [min(Y.min(), Y_pred_mult.min()), max(Y.max(), Y_pred_mult.max())]
plt.plot(limite, limite, 'r--', linewidth=2, label='Predicción perfecta (referencia)')

# Puntos: real vs predicho
plt.scatter(Y, Y_pred_mult, color='steelblue', s=90, zorder=5, label='Predicciones del modelo')

# Líneas de residuo
plt.vlines(Y, Y, Y_pred_mult, colors='gray', linewidth=0.8, alpha=0.5, label='Residuos')

plt.xlabel('Matrícula Universitaria REAL (%)', fontsize=12)
plt.ylabel('Matrícula Universitaria PREDICHA (%)', fontsize=12)
plt.title(f'Modelo Múltiple — Valores Reales vs Predichos\nR² = {r2_mult:.4f}', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Cuanto más cerca estén los puntos de la línea roja, mejor es el modelo.')

## Paso 8 — Predicciones con el modelo múltiple

Para predecir con el modelo múltiple necesitamos proporcionar un valor para **cada una** de las tres variables.

In [ ]:
# Escenarios: cada fila tiene [Gasto_%PIB, PIB_per_Capita, Becas_miles]
escenarios = [
    [2.0, 1800,  11.0],
    [2.5, 2500,  14.0],
    [3.5, 4000,  22.0],
    [4.5, 6000,  35.0],
    [5.5, 8000,  42.0],
    [6.2, 10000, 45.0],
]
nombres = [
    'Crisis (actual)',
    'Recuperación leve',
    'Recuperación moderada',
    'Pre-crisis (2014)',
    'Nivel histórico (2010)',
    'Máximo histórico (2007)',
]

print('Predicciones del modelo múltiple:')
print(f'{"Gasto":>6} | {"PIB/cap":>8} | {"Becas":>6} | {"Matrícula pred.":>16} | Escenario')
print('-' * 72)

preds_mult = []
for esc, nom in zip(escenarios, nombres):
    pred = b0_m + b1_m * esc[0] + b2_m * esc[1] + b3_m * esc[2]
    preds_mult.append(pred)
    print(f'{esc[0]:>6.1f}% | {esc[1]:>8,} | {esc[2]:>6.1f} | {pred:>14.2f}%  | {nom}')

In [ ]:
# Gráfico de predicciones múltiple
gastos_esc = [e[0] for e in escenarios]

plt.figure(figsize=(10, 6))
plt.scatter(df_limpio['Gasto_Educacion_%PIB'], Y, color='steelblue', s=80, label='Datos reales', zorder=5)
plt.scatter(gastos_esc, preds_mult, color='green', s=150, marker='D', zorder=6, label='Predicciones modelo múltiple')

plt.xlabel('Gasto en Educación (% PIB)', fontsize=12)
plt.ylabel('Matrícula Universitaria (%)', fontsize=12)
plt.title('Predicciones — Modelo de Regresión Múltiple\nVenezuela: Escenarios de Política Educativa', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# 🟠 PARTE B — Regresión Logarítmica

## ¿Qué es la Regresión Logarítmica?

A veces la relación entre dos variables **no es una línea recta**, sino una **curva**.  
Por ejemplo: al principio, aumentar el PIB genera grandes saltos en la matrícula,  
pero cuando el PIB ya es alto, el efecto se va reduciendo.

Eso es exactamente lo que describe el **logaritmo**:

$$\hat{Y} = \beta_0 + \beta_1 \cdot \ln(X)$$

El truco es sencillo: **transformamos X aplicando logaritmo** y luego hacemos una regresión lineal normal sobre esa versión transformada.

### ¿Cuándo usar la regresión logarítmica?
- Cuando la variable X tiene valores muy grandes (como el PIB en dólares)
- Cuando el gráfico de dispersión muestra una **curva**, no una línea
- Cuando los incrementos al inicio son grandes pero se van reduciendo

### Variable que usaremos
- **X** = PIB per Cápita (USD)
- **Y** = Matrícula Universitaria (%)

## Paso 9 — Verificar si la relación es curva o recta

Antes de aplicar la transformación logarítmica, veamos el gráfico de dispersión.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

X_pib = df_limpio['PIB_per_Capita_USD'].values
Y_mat = df_limpio['Matricula_Universitaria_%'].values

# Gráfico 1: X original vs Y
axes[0].scatter(X_pib, Y_mat, color='steelblue', s=80)
axes[0].set_xlabel('PIB per Cápita (USD)', fontsize=11)
axes[0].set_ylabel('Matrícula Universitaria (%)', fontsize=11)
axes[0].set_title('PIB vs Matrícula\n(escala original)', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Gráfico 2: ln(X) vs Y
X_log = np.log(X_pib)
axes[1].scatter(X_log, Y_mat, color='darkorange', s=80)
axes[1].set_xlabel('ln(PIB per Cápita)', fontsize=11)
axes[1].set_ylabel('Matrícula Universitaria (%)', fontsize=11)
axes[1].set_title('ln(PIB) vs Matrícula\n(escala logarítmica — más lineal)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('¿Curva o línea recta? Comparación de escalas', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Observación:')
print('  Gráfico izquierdo → la nube de puntos forma una curva.')
print('  Gráfico derecho   → al aplicar ln(X), los puntos se alinean mejor.')
print('  Esto confirma que la regresión logarítmica es más adecuada aquí.')

## Paso 10 — Construcción del modelo logarítmico

### Matemáticamente

El modelo es:
$$\hat{Y} = \beta_0 + \beta_1 \cdot \ln(X)$$

Para calcularlo:
1. Creamos una nueva variable $Z = \ln(X)$
2. Ajustamos una regresión lineal simple entre $Z$ e $Y$
3. Los coeficientes se obtienen con las mismas fórmulas de siempre:

$$\beta_1 = \frac{\sum(z_i - \bar{z})(y_i - \bar{y})}{\sum(z_i - \bar{z})^2} \qquad \beta_0 = \bar{y} - \beta_1 \cdot \bar{z}$$

In [ ]:
# Paso 1: transformar X aplicando logaritmo natural
X_log = np.log(X_pib)

# Paso 2: calcular los coeficientes con la fórmula matemática
z_media = X_log.mean()
y_media = Y_mat.mean()

b1_log = np.sum((X_log - z_media) * (Y_mat - y_media)) / np.sum((X_log - z_media) ** 2)
b0_log = y_media - b1_log * z_media

print('Cálculo manual — modelo logarítmico:')
print(f'  Media de ln(X) : {z_media:.4f}')
print(f'  Media de Y     : {y_media:.4f}')
print(f'  β₁             = {b1_log:.4f}')
print(f'  β₀             = {b0_log:.4f}')
print()
print(f'Ecuación del modelo:')
print(f'  Ŷ = {b0_log:.4f} + {b1_log:.4f} · ln(X)')

In [ ]:
# Verificación con scikit-learn
modelo_log = LinearRegression()
modelo_log.fit(X_log.reshape(-1, 1), Y_mat)

b0_log_sk = modelo_log.intercept_
b1_log_sk = modelo_log.coef_[0]

Y_pred_log = modelo_log.predict(X_log.reshape(-1, 1))
r2_log = r2_score(Y_mat, Y_pred_log)

print('Verificación con scikit-learn:')
print(f'  β₀ = {b0_log_sk:.4f}  ← igual al cálculo manual ✓')
print(f'  β₁ = {b1_log_sk:.4f}  ← igual al cálculo manual ✓')
print()
print(f'  R² = {r2_log:.4f}')
print(f'  El modelo logarítmico explica el {r2_log*100:.1f}% de la variabilidad.')
print()
print('Interpretación:')
print(f'  Cuando el PIB per cápita aumenta un 1%, la matrícula universitaria')
print(f'  sube aproximadamente {b1_log_sk/100:.4f} puntos porcentuales.')
print(f'  (Nota: en logaritmos, un cambio del 1% en X equivale a ln(1.01) ≈ 0.01)')

## Paso 11 — Gráfico del modelo logarítmico

El gráfico muestra:
- 🟠 **Puntos naranjas:** datos reales de cada año
- 🔴 **Curva roja:** la curva del modelo logarítmico
- ⬛ **Líneas grises:** residuos (distancia entre cada dato real y la curva)

In [ ]:
# Generamos la curva suavizada para graficar
x_curva = np.linspace(X_pib.min(), X_pib.max(), 300)
y_curva = b0_log_sk + b1_log_sk * np.log(x_curva)

plt.figure(figsize=(10, 7))

# Residuos
plt.vlines(X_pib, Y_mat, Y_pred_log, colors='gray', linewidth=1, alpha=0.5, label='Residuos (distancia)')

# Datos reales
plt.scatter(X_pib, Y_mat, color='darkorange', s=90, zorder=5, label='Datos reales')

# Curva del modelo
plt.plot(x_curva, y_curva, color='red', linewidth=2.5, label=f'Curva: Ŷ = {b0_log_sk:.2f} + {b1_log_sk:.2f}·ln(X)')

plt.xlabel('PIB per Cápita (USD)', fontsize=12)
plt.ylabel('Matrícula Universitaria (%)', fontsize=12)
plt.title('Modelo de Regresión Logarítmica\nPIB per Cápita → Matrícula Universitaria | Venezuela 2000–2023',
          fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

texto = f'Ŷ = {b0_log_sk:.3f} + {b1_log_sk:.3f}·ln(X)\nR² = {r2_log:.4f}'
plt.text(0.55, 0.15, texto, transform=plt.gca().transAxes,
         fontsize=11, verticalalignment='bottom',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

## Paso 12 — Predicciones con el modelo logarítmico

Para predecir con el modelo logarítmico usamos la ecuación:

$$\hat{Y} = \beta_0 + \beta_1 \cdot \ln(X)$$

El único cambio respecto al modelo simple es que aplicamos `np.log()` a X antes de calcular.

In [ ]:
pibs = [1500, 2000, 3000, 5000, 7000, 9000, 11000, 13000]
descripciones_log = [
    'Crisis profunda',
    'Nivel actual (2023)',
    'Recuperación baja',
    'Recuperación media',
    'Pre-crisis (2015)',
    'Nivel histórico (2009)',
    'Pico histórico (2012)',
    'Escenario optimista',
]

print('Predicciones del modelo logarítmico:')
print(f'{"PIB per cápita":>15} | {"ln(PIB)":>8} | {"Matrícula pred.":>16} | Escenario')
print('-' * 68)

preds_log = []
for pib, desc in zip(pibs, descripciones_log):
    ln_pib = np.log(pib)
    pred   = b0_log_sk + b1_log_sk * ln_pib
    preds_log.append(pred)
    print(f'{pib:>15,} | {ln_pib:>8.4f} | {pred:>14.2f}%  | {desc}')

In [ ]:
# Gráfico de predicciones logarítmico
plt.figure(figsize=(10, 6))

plt.plot(x_curva, y_curva, color='red', linewidth=2, alpha=0.7, label='Curva del modelo')
plt.scatter(X_pib, Y_mat, color='darkorange', s=80, zorder=5, label='Datos reales (2000–2023)')
plt.scatter(pibs, preds_log, color='green', s=150, marker='D', zorder=6, label='Predicciones por escenario')

plt.xlabel('PIB per Cápita (USD)', fontsize=12)
plt.ylabel('Matrícula Universitaria (%)', fontsize=12)
plt.title('Predicciones — Modelo de Regresión Logarítmica', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

texto = f'Ŷ = {b0_log_sk:.3f} + {b1_log_sk:.3f}·ln(X)\nR² = {r2_log:.4f}'
plt.text(0.55, 0.15, texto, transform=plt.gca().transAxes,
         fontsize=11, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

## Paso 13 — Comparación final de los tres modelos

Comparamos los tres modelos que hemos visto en estos notebooks.

In [ ]:
print('Comparación de los tres modelos:')
print()
print(f'  Modelo simple    → Ŷ = β₀ + β₁·X          R² = {r2_simple:.4f}  ({r2_simple*100:.1f}%)')
print(f'  Modelo múltiple  → Ŷ = β₀ + β₁X₁+β₂X₂+β₃X₃  R² = {r2_mult:.4f}  ({r2_mult*100:.1f}%)')
print(f'  Modelo logarítml → Ŷ = β₀ + β₁·ln(X)      R² = {r2_log:.4f}  ({r2_log*100:.1f}%)')
print()
mejor_r2 = max(r2_simple, r2_mult, r2_log)
nombres_modelos = ['simple', 'múltiple', 'logarítmico']
mejor_nombre = nombres_modelos[[r2_simple, r2_mult, r2_log].index(mejor_r2)]
print(f'  → El modelo con mejor R² es el {mejor_nombre} (R² = {mejor_r2:.4f})')

In [ ]:
# Gráfico de barras comparando R² de los tres modelos
modelos   = ['Regresión\nSimple', 'Regresión\nMúltiple', 'Regresión\nLogarítmica']
r2_valores = [r2_simple, r2_mult, r2_log]
colores    = ['steelblue', 'green', 'darkorange']

plt.figure(figsize=(8, 5))
barras = plt.bar(modelos, r2_valores, color=colores, width=0.5, edgecolor='white')

# Etiquetas encima de cada barra
plt.bar_label(barras, labels=[f'{v:.4f}' for v in r2_valores], fontsize=12, fontweight='bold', padding=4)

plt.ylim(0, 1.05)
plt.ylabel('Coeficiente de Determinación R²', fontsize=12)
plt.title('Comparación de R² — Los Tres Modelos', fontsize=13, fontweight='bold')
plt.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5, label='Umbral R² = 0.90')
plt.legend(fontsize=10)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Paso 14 — Ejercicios propuestos

Completa cada celda reemplazando los `???` con el código correcto.

In [ ]:
# EJERCICIO 1 — Regresión Múltiple
# El gobierno planea: Gasto = 3.8% PIB, PIB per cápita = 4.500 USD, Becas = 28.000
# ¿Qué matrícula predice el modelo múltiple?

gasto   = ???
pib_cap = ???
becas   = ???

pred_mult = b0_m + b1_m * gasto + b2_m * pib_cap + b3_m * becas
print(f'Matrícula predicha por el modelo múltiple: {pred_mult:.2f}%')

In [ ]:
# EJERCICIO 2 — Regresión Logarítmica
# Si el PIB per cápita llega a 6.500 USD,
# ¿qué matrícula universitaria predice el modelo logarítmico?
# Pista: Ŷ = β₀ + β₁ · ln(X)  → usa np.log()

pib_nuevo = ???
pred_log  = b0_log_sk + b1_log_sk * np.log(???)
print(f'Con un PIB per cápita de {pib_nuevo} USD, la matrícula predicha es: {pred_log:.2f}%')

In [ ]:
# EJERCICIO 3 — Comparar modelos
# Calcula el R² del modelo logarítmico usando r2_score()
# y compáralo con el R² del modelo múltiple
# ¿Cuál es mayor?

# Ya están calculados:
print(f'R² modelo múltiple   : {???:.4f}')
print(f'R² modelo logarítmico: {???:.4f}')

if ??? > ???:
    print('El modelo múltiple tiene mejor ajuste')
else:
    print('El modelo logarítmico tiene mejor ajuste')

In [ ]:
# EJERCICIO 4 — Interpretación
# Completa las frases con los valores correctos de los coeficientes

print('Completa la interpretación del modelo múltiple:')
print()
print(f'  β₁ = {b1_m:.4f}')
print(f'  → Por cada 1% adicional del PIB en educación, la matrícula sube ??? puntos.')
print()
print(f'  β₃ = {b3_m:.4f}')
print(f'  → Por cada 1.000 becas adicionales, la matrícula sube ??? puntos.')

## Conclusiones

### ¿Qué aprendimos con este análisis?

**1. Sobre la Regresión Múltiple**  
Agregar más variables al modelo generalmente mejora su capacidad predictiva. En este caso, combinar el gasto en educación, el PIB per cápita y las becas otorgadas nos da un R² superior al del modelo simple. Esto confirma que la matrícula universitaria en Venezuela **no depende de un solo factor**, sino de varios simultáneamente.

Cada coeficiente nos dice cuánto aporta su variable **de forma independiente**: por ejemplo, aumentar las becas mejora la matrícula incluso si el gasto en educación permanece igual.

**2. Sobre la Regresión Logarítmica**  
La relación entre el PIB per cápita y la matrícula universitaria **no es lineal**: el efecto del PIB es más fuerte en niveles bajos y se va reduciendo a medida que el PIB crece. Esto es exactamente lo que modela el logaritmo. Al aplicar `ln(X)` convertimos esa curva en una relación lineal que podemos ajustar con las mismas herramientas de siempre.

**3. ¿Cuándo usar cada modelo?**

| Modelo | Úsalo cuando... |
|---|---|
| Regresión simple | Tienes una sola variable predictora y la relación es lineal |
| Regresión múltiple | Tienes varias variables predictoras relevantes |
| Regresión logarítmica | La relación es curva y se aplana al crecer X |

**4. Lección de política pública**  
Los datos de Venezuela muestran que el acceso universitario está profundamente vinculado al nivel económico y a las decisiones de inversión en educación. La crisis 2016–2020 no solo redujo el PIB: también contrajo las becas, el gasto educativo y, como consecuencia directa, la cantidad de jóvenes que pudieron acceder a la universidad. Los tres modelos apuntan en la misma dirección: **recuperar la inversión educativa es la variable con mayor impacto directo y medible.**